In [1]:
import numpy as np
import gymnasium as gym

In [2]:
class FourierBasis:
    def __init__(self, state_dim, order):
        self.state_dim = state_dim
        self.order = order
        self.num_features = (order + 1) ** state_dim
        self.c = np.array(np.meshgrid(*[range(order + 1)] * state_dim)).T.reshape(-1, state_dim)

    def get_features(self, state):
        # Assumes state is normalized to [0, 1]
        return np.cos(np.pi * np.dot(self.c, state))

In [3]:
class SarsaLambdaAgent:
    def __init__(self, env, order=3, alpha=0.005, gamma=0.99, lam=0.5, epsilon=0.1):
        self.env = env
        self.state_dim = env.observation_space.shape[0]
        self.num_actions = env.action_space.n
        self.basis = FourierBasis(self.state_dim, order)
        self.num_features = self.basis.num_features
        self.alpha = alpha
        self.gamma = gamma
        self.lam = lam
        self.epsilon = epsilon
        self.weights = np.zeros((self.num_actions, self.num_features))
        self.xbar = np.zeros((2, 2))
        self.xbar[0, :] = env.observation_space.low
        self.xbar[1, :] = env.observation_space.high

    def normalize_state(self, state):
        # Normalize to [0, 1] using environment bounds
        low = self.env.observation_space.low
        high = self.env.observation_space.high
        return (state - low) / (high - low)

    def get_q(self, state):
        features = self.basis.get_features(self.normalize_state(state))
        return np.dot(self.weights, features), features

    def choose_action(self, state):
        q_values, _ = self.get_q(state)
        if np.random.rand() < self.epsilon:
            return self.env.action_space.sample()
        else:
            return np.argmax(q_values)

    def train(self, num_episodes=500, num_timesteps=600):
        for episode in range(num_episodes):
            state = self.env.reset()[0]
            action = self.choose_action(state)
            q_values, features = self.get_q(state)
            e = np.zeros((self.num_actions, self.num_features))  # eligibility traces
            rewards = 0
            done = False
            while not done:
                next_state, reward, done, _, info = self.env.step(action)
                rewards += reward
                next_action = self.choose_action(next_state)
                next_q_values, next_features = self.get_q(next_state)

                delta = reward + self.gamma * next_q_values[next_action] * (not done) - q_values[action]
                e *= self.gamma * self.lam
                e[action] += features  # accumulating traces

                self.weights += self.alpha * delta * e

                # Update state, action, q_values, features
                state, action,q_values, features  = next_state, next_action, next_q_values, next_features
  

            if (episode + 1) % 50 == 0:
                print(f"Episode {episode + 1} complete with {rewards} as reward.")


    


In [4]:
env = gym.make("MountainCar-v0")
agent = SarsaLambdaAgent(env, order=3, alpha=0.01, gamma=1.0, lam=0.9, epsilon=0.1)
agent.train(num_episodes=500, num_timesteps=600)

Episode 50 complete with -197.0 as reward.
Episode 100 complete with -150.0 as reward.
Episode 150 complete with -170.0 as reward.
Episode 200 complete with -147.0 as reward.
Episode 250 complete with -172.0 as reward.
Episode 300 complete with -148.0 as reward.
Episode 350 complete with -148.0 as reward.
Episode 400 complete with -115.0 as reward.
Episode 450 complete with -135.0 as reward.
Episode 500 complete with -197.0 as reward.
